# LSTM Gesture Classification — Model Variations

This notebook trains **5 distinct LSTM configurations** for the Data Glove gesture classification project.

| # | Model | Input Dim | Normalization |
|---|-------|-----------|---------------|
| 1 | Raw IMU | 6 | ❌ |
| 2 | Raw IMU | 6 | ✅ |
| 3 | Madgwick Quaternions | 4 | ❌ |
| 4 | Raw + Madgwick | 10 | ❌ |
| 5 | Raw + Madgwick | 10 | ✅ |

In [ ]:
import os
import json
import math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score, confusion_matrix
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import ahrs

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ============================================================
# CONFIGS — edit these as needed
# ============================================================
DATA_PATH     = "Database/img/"
MODELS_DIR    = "models/"
SEQ_LEN       = 300        # 3 s @ 10 ms = 300 steps
BATCH_SIZE    = 32
HIDDEN_DIM    = 128
NUM_LAYERS    = 1
DROPOUT       = 0.0
EPOCHS        = 90
LR            = 1e-3
TEST_SIZE     = 0.2
RANDOM_STATE  = 42

# MPU6050 raw‑ADC scaling (change if your FS config differs)
ACCEL_SCALE   = 16384.0    # ±2 g
GYRO_SCALE    = 131.0      # ±250 °/s
SAMPLE_RATE   = 100.0      # Hz  (10 ms period)

os.makedirs(MODELS_DIR, exist_ok=True)
print("Config loaded. Models will be saved to:", MODELS_DIR)

## Madgwick Filter — Quaternion Mathematics

### What is a Quaternion?

A **quaternion** is a 4‑dimensional hyper‑complex number used to represent 3‑D rotations without gimbal lock:

$$q = q_0 + q_1\,\mathbf{i} + q_2\,\mathbf{j} + q_3\,\mathbf{k}$$

where $q_0$ is the scalar (real) part and $(q_1, q_2, q_3)$ is the vector (imaginary) part.  
A **unit quaternion** satisfies $\|q\| = \sqrt{q_0^2 + q_1^2 + q_2^2 + q_3^2} = 1$.

### Quaternion Multiplication (Hamilton Product)

Given two quaternions $p$ and $r$:

$$
p \otimes r = \begin{bmatrix}
p_0 r_0 - p_1 r_1 - p_2 r_2 - p_3 r_3 \\
p_0 r_1 + p_1 r_0 + p_2 r_3 - p_3 r_2 \\
p_0 r_2 - p_1 r_3 + p_2 r_0 + p_3 r_1 \\
p_0 r_3 + p_1 r_2 - p_2 r_1 + p_3 r_0
\end{bmatrix}
$$

### Madgwick Filter Update Equations

**Step 1 — Gyroscope integration (orientation rate):**

$$\dot{q}_{\omega,t} = \tfrac{1}{2}\, q_{t-1} \otimes \begin{bmatrix} 0 \\ \omega_x \\ \omega_y \\ \omega_z \end{bmatrix}$$

**Step 2 — Objective function from accelerometer:**

The accelerometer measures the gravity vector in the sensor frame. The objective function quantifies the error between the measured acceleration $\hat{a}$ and the expected gravity direction rotated by $q$:

$$f(q, \hat{a}) = q^* \otimes \begin{bmatrix} 0\\0\\0\\1 \end{bmatrix} \otimes q - \hat{a}$$

Expanded:

$$f(q, \hat{a}) = \begin{bmatrix}
2(q_1 q_3 - q_0 q_2) - a_x \\
2(q_0 q_1 + q_2 q_3) - a_y \\
2(\tfrac{1}{2} - q_1^2 - q_2^2) - a_z
\end{bmatrix}$$

**Step 3 — Jacobian:**

$$J(q) = \begin{bmatrix}
-2q_2 & 2q_3 & -2q_0 & 2q_1 \\
 2q_1 & 2q_0 &  2q_3 & 2q_2 \\
 0    & -4q_1 & -4q_2 & 0
\end{bmatrix}$$

**Step 4 — Gradient descent correction:**

$$\nabla f = J^T \cdot f$$

**Step 5 — Fuse gyroscope and accelerometer:**

$$q_t = q_{t-1} + \left( \dot{q}_{\omega,t} - \beta \frac{\nabla f}{\|\nabla f\|} \right) \Delta t$$

where $\beta$ is the filter gain representing the estimated gyroscope measurement error,  
and $\Delta t = 1 / f_s = 0.01\,\text{s}$ for our 100 Hz sampling rate.

The result is normalised to maintain $\|q_t\| = 1$.

## Data Loading & Preprocessing

In [ ]:
def load_and_preprocess_data():
    """Load CSV files, scale raw ADC values, compute Madgwick quaternions.
    
    Returns
    -------
    X_raw  : np.ndarray  [N, SEQ_LEN, 6]  — scaled accel (m/s²) + gyro (rad/s)
    X_madg : np.ndarray  [N, SEQ_LEN, 4]  — quaternions (q0, q1, q2, q3)
    y      : np.ndarray  [N]               — integer labels
    label_to_idx : dict
    """
    raw_sequences = []
    madg_sequences = []
    labels = []

    madgwick = ahrs.filters.Madgwick(frequency=SAMPLE_RATE)

    for filename in sorted(os.listdir(DATA_PATH)):
        if not filename.endswith('.csv'):
            continue

        path = os.path.join(DATA_PATH, filename)
        df = pd.read_csv(path, header=None, dtype=str)

        # Clean non‑numeric characters, convert
        df = df.astype(str).apply(
            lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x)
        )
        df = df.replace(r'[^\d\.-]', '', regex=True)
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna()

        # Pad / truncate to SEQ_LEN
        if df.shape[0] >= SEQ_LEN:
            df = df.iloc[:SEQ_LEN]
        else:
            pad = pd.DataFrame(
                np.zeros((SEQ_LEN - df.shape[0], df.shape[1]))
            )
            df = pd.concat([df, pad], ignore_index=True)

        raw = df.values.astype(np.float32)  # [300, 6]

        # --- Scale to SI units ---
        accel_g    = raw[:, 0:3] / ACCEL_SCALE          # in g
        accel_ms2  = accel_g * 9.80665                   # m/s²
        gyro_dps   = raw[:, 3:6] / GYRO_SCALE           # °/s
        gyro_rads  = np.deg2rad(gyro_dps)                # rad/s

        raw_scaled = np.concatenate([accel_ms2, gyro_rads], axis=1)  # [300, 6]
        raw_sequences.append(raw_scaled)

        # --- Madgwick quaternions ---
        Q = np.zeros((SEQ_LEN, 4), dtype=np.float64)
        Q[0] = [1.0, 0.0, 0.0, 0.0]
        for t in range(1, SEQ_LEN):
            Q[t] = madgwick.updateIMU(Q[t - 1], gyr=gyro_rads[t], acc=accel_g[t])

        madg_sequences.append(Q.astype(np.float32))
        labels.append(filename.split('_')[0])

    label_to_idx = {lbl: idx for idx, lbl in enumerate(sorted(set(labels)))}
    y_indices = np.array([label_to_idx[lbl] for lbl in labels])

    X_raw  = np.stack(raw_sequences)   # [N, 300, 6]
    X_madg = np.stack(madg_sequences)   # [N, 300, 4]

    return X_raw, X_madg, y_indices, label_to_idx


print("Loading dataset …")
X_raw, X_madg, y, label_to_idx = load_and_preprocess_data()
idx_to_label = {v: k for k, v in label_to_idx.items()}

print(f"\nSamples loaded : {len(y)}")
print(f"Classes        : {label_to_idx}")
print(f"X_raw  shape   : {X_raw.shape}")
print(f"X_madg shape   : {X_madg.shape}")
print(f"y      shape   : {y.shape}")

## Model Definition & Training Pipeline

In [ ]:
# ────────────────────────────────────────────────────────────────
# LSTM Classifier
# ────────────────────────────────────────────────────────────────
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim,
                 num_layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])  # last layer hidden state


# ────────────────────────────────────────────────────────────────
# Dataset helper
# ────────────────────────────────────────────────────────────────
class GestureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
def train_and_evaluate(X, y, input_dim, model_name, normalize=False):
    """
    Full training + evaluation pipeline.
    
    Saves:
      - models/<model_name>.pth
      - models/<model_name>_metrics.json
    
    Prints:
      - Shape info, per‑epoch loss, final classification report
    
    Plots:
      - Train / Test loss curve
      - Train / Test accuracy curve
      - F1 score curve
      - Confusion matrix heatmap
    """
    print(f"\n{'=' * 60}")
    print(f"  MODEL: {model_name}")
    print(f"  Normalize: {normalize}  |  Input dim: {input_dim}")
    print(f"{'=' * 60}")
    print(f"  X shape : {X.shape}")
    print(f"  y shape : {y.shape}")

    # ── Split ──────────────────────────────────────────────────
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    # ── Normalize (per‑feature, fit on train only) ─────────────
    if normalize:
        mu  = X_train.mean(axis=(0, 1), keepdims=True)
        std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8
        X_train = (X_train - mu) / std
        X_test  = (X_test  - mu) / std
        print(f"  Norm μ  : {mu.squeeze()}")
        print(f"  Norm σ  : {std.squeeze()}")

    print(f"  X_train : {X_train.shape}  |  X_test : {X_test.shape}")
    print()

    # ── Loaders ────────────────────────────────────────────────
    train_loader = DataLoader(
        GestureDataset(X_train, y_train),
        batch_size=BATCH_SIZE, shuffle=True,
    )
    test_loader = DataLoader(
        GestureDataset(X_test, y_test),
        batch_size=BATCH_SIZE,
    )

    # ── Model ──────────────────────────────────────────────────
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(label_to_idx)
    model = LSTMClassifier(
        input_dim, HIDDEN_DIM, num_classes,
        num_layers=NUM_LAYERS, dropout=DROPOUT,
    ).to(device)

    print(f"  Device  : {device}")
    print(f"  Params  : {sum(p.numel() for p in model.parameters()):,}")
    print()

    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()

    # ── Training loop ──────────────────────────────────────────
    history = {
        "train_loss": [], "test_loss": [],
        "train_acc": [],  "test_acc": [],
        "test_f1": [],
    }

    for epoch in range(1, EPOCHS + 1):
        # --- train ---
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()
            run_loss += loss.item()
            correct += (out.argmax(1) == by).sum().item()
            total   += by.size(0)

        train_loss = run_loss / len(train_loader)
        train_acc  = correct / total

        # --- eval ---
        model.eval()
        run_loss_t, correct_t, total_t = 0.0, 0, 0
        all_preds, all_targets = [], []
        with torch.no_grad():
            for bx, by in test_loader:
                bx, by = bx.to(device), by.to(device)
                out = model(bx)
                run_loss_t += criterion(out, by).item()
                preds = out.argmax(1)
                correct_t += (preds == by).sum().item()
                total_t   += by.size(0)
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(by.cpu().numpy())

        test_loss = run_loss_t / len(test_loader)
        test_acc  = correct_t / total_t
        f1 = f1_score(all_targets, all_preds, average='weighted')

        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["test_f1"].append(f1)

        # Print every 10 epochs + last
        if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
            print(
                f"  Epoch {epoch:3d}/{EPOCHS}  "
                f"TrainLoss={train_loss:.4f}  TestLoss={test_loss:.4f}  "
                f"TrainAcc={train_acc:.4f}  TestAcc={test_acc:.4f}  "
                f"F1={f1:.4f}"
            )

    # ── Save model & metrics ───────────────────────────────────
    torch.save(model.state_dict(), os.path.join(MODELS_DIR, f"{model_name}.pth"))
    with open(os.path.join(MODELS_DIR, f"{model_name}_metrics.json"), 'w') as fp:
        json.dump(history, fp)

    # ── Final report ───────────────────────────────────────────
    target_names = [idx_to_label[i] for i in range(num_classes)]
    print(f"\n  Final Test Accuracy: {test_acc:.4f}")
    print(f"  Final F1 (weighted): {f1:.4f}")
    print()
    print(classification_report(all_targets, all_preds, target_names=target_names))

    # ── Plots ──────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"{model_name}", fontsize=16, fontweight='bold')

    epochs_range = range(1, EPOCHS + 1)

    # 1) Loss
    axes[0, 0].plot(epochs_range, history['train_loss'], label='Train')
    axes[0, 0].plot(epochs_range, history['test_loss'],  label='Test')
    axes[0, 0].set_title('Loss per Epoch')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # 2) Accuracy
    axes[0, 1].plot(epochs_range, history['train_acc'], label='Train')
    axes[0, 1].plot(epochs_range, history['test_acc'],  label='Test')
    axes[0, 1].set_title('Accuracy per Epoch')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3) F1
    axes[1, 0].plot(epochs_range, history['test_f1'], color='green', label='Test F1')
    axes[1, 0].set_title('F1 Score (weighted) per Epoch')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4) Confusion matrix
    cm = confusion_matrix(all_targets, all_preds)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=target_names, yticklabels=target_names,
        ax=axes[1, 1],
    )
    axes[1, 1].set_title('Confusion Matrix (final epoch)')
    axes[1, 1].set_xlabel('Predicted')
    axes[1, 1].set_ylabel('True')

    plt.tight_layout()
    plt.show()

    print(f"  ✓ Saved: {MODELS_DIR}{model_name}.pth")
    print(f"  ✓ Saved: {MODELS_DIR}{model_name}_metrics.json")

    return history

## Wrapper Functions

In [ ]:
def train_6_input_model(X_raw, y, normalize=False):
    """Train LSTM on the 6 raw IMU channels.
    
    Use Cases:
      normalize=False → Model 1 (6 raw)
      normalize=True  → Model 2 (6 norm)
    """
    name = "model_2_6_norm" if normalize else "model_1_6_raw"
    return train_and_evaluate(X_raw, y, input_dim=6,
                             model_name=name, normalize=normalize)


def train_4_input_model(X_madg, y, normalize=False):
    """Train LSTM on 4 Madgwick quaternion channels.
    
    Use Case:
      normalize=False → Model 3 (4 madgwick)
    """
    name = "model_3_4_madg"
    return train_and_evaluate(X_madg, y, input_dim=4,
                             model_name=name, normalize=normalize)


def train_10_input_model(X_raw, X_madg, y, normalize=False):
    """Train LSTM on 10 channels (6 raw + 4 quaternion).
    
    Use Cases:
      normalize=False → Model 4 (10 raw)
      normalize=True  → Model 5 (10 norm)
    """
    X_concat = np.concatenate([X_raw, X_madg], axis=2)  # [N, 300, 10]
    name = "model_5_10_norm" if normalize else "model_4_10_raw"
    return train_and_evaluate(X_concat, y, input_dim=10,
                             model_name=name, normalize=normalize)

---
## Model 1 — LSTM with 6 Raw Inputs (ax, ay, az, gx, gy, gz)

In [ ]:
h1 = train_6_input_model(X_raw, y, normalize=False)

---
## Model 2 — LSTM with 6 Normalized Inputs

In [ ]:
h2 = train_6_input_model(X_raw, y, normalize=True)

---
## Model 3 — LSTM with 4 Madgwick Quaternion Inputs

In [ ]:
h3 = train_4_input_model(X_madg, y, normalize=False)

---
## Model 4 — LSTM with 10 Inputs (Raw + Madgwick)

In [ ]:
h4 = train_10_input_model(X_raw, X_madg, y, normalize=False)

---
## Model 5 — LSTM with 10 Normalized Inputs (Raw + Madgwick)

In [ ]:
h5 = train_10_input_model(X_raw, X_madg, y, normalize=True)

---
## Quick Side‑by‑Side Summary

In [ ]:
# Quick summary table
names = [
    "1) 6 Raw",
    "2) 6 Norm",
    "3) 4 Madgwick",
    "4) 10 Raw",
    "5) 10 Norm",
]
histories = [h1, h2, h3, h4, h5]

print(f"{'Model':<18} {'Best Test Acc':>14} {'Best F1':>10} {'Final Train Loss':>18}")
print("-" * 62)
for name, h in zip(names, histories):
    best_acc = max(h['test_acc'])
    best_f1  = max(h['test_f1'])
    final_tl = h['train_loss'][-1]
    print(f"{name:<18} {best_acc:>14.4f} {best_f1:>10.4f} {final_tl:>18.4f}")

# Overlay plot: Test accuracy across all models
plt.figure(figsize=(12, 5))
for name, h in zip(names, histories):
    plt.plot(h['test_acc'], label=name)
plt.title('Test Accuracy — All Models')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ All 5 models trained. Weights and metrics saved to:", MODELS_DIR)